# Proyecto CPV Farma — Análogo al "Proyecto Music"
### De la verificación continua del proceso (CPV) a la ciencia de datos

**Contexto:** Trabajas como químico de documentación, enfocado en verificación continua
del proceso de fabricación (CPV) y hoy analizas estadísticamente esos datos (control
estadístico de proceso, capacidad, etc.). Este ejercicio es el **"nivel 1"** hacia un
análisis más propio de ciencia de datos: exploración, limpieza, agregaciones y prueba
de hipótesis con `pandas`, tal como el "Proyecto Music" de TripleTen (que compara
hábitos de escucha entre dos ciudades), pero aplicado a dos líneas de producción.

| Proyecto Music (original) | Proyecto CPV Farma (este ejercicio) |
|---|---|
| `userID` (usuario) | `lote_ID` (lote de fabricación) |
| `Track` (canción) | `prueba` (parámetro/prueba de calidad) |
| `artist` (artista) | `Equipo` (equipo/instrumento usado) |
| `genre` (género musical) | `resultado` (Conforme / Alerta / OOS / OOT) |
| `City` (Springfield / Shelbyville) | `Linea_produccion` (Línea A / Línea B) |
| `time` (hora de reproducción) | `hora_muestreo` (hora de muestreo) |
| `Day` (día de la semana) | `Dia` (día de la semana) |

**Dataset:** `cpv_farma_dataset.csv` (1,515 registros, con nombres de columnas sucios,
valores ausentes y duplicados **a propósito**, igual que el dataset original).

---
## Plan de trabajo para 2 horas (puedes ajustarlo)

| Bloque | Tiempo | Contenido |
|---|---|---|
| 1 | 10 min | Entorno: verificar librerías, cargar el CSV |
| 2 | 25 min | Etapa 1 — Descripción de los datos (`head`, `info`, `describe`) |
| 3 | 40 min | Etapa 2 — Preprocesamiento (nombres de columnas, ausentes, duplicados, tipos) |
| 4 | 35 min | Etapa 3 — Primeras hipótesis (H1: actividad por día/línea) |
| 5 | 10 min | Documentar hallazgos preliminares y dejar pendientes H2 y H3 para la próxima sesión |

> Objetivo realista de hoy: dejar documentadas las Etapas 1 y 2 completas, y arrancar la H1
> de la Etapa 3. Las hipótesis H2 y H3 quedan listas (con preguntas guía) para la siguiente sesión.


## Objetivo del proyecto y las 3 hipótesis a probar

**Pregunta de negocio:** ¿El comportamiento del proceso (actividad de muestreo y tipo de
resultado) difiere de forma significativa entre la Línea A y la Línea B, según el día de
la semana y el turno?

**Hipótesis (equivalentes a las del Proyecto Music):**

- **H1:** La actividad de muestreo/pruebas difiere según el día de la semana y la línea
  de producción (¿hay más pruebas los lunes? ¿una línea genera más registros que otra?).
- **H2:** Los lunes por la mañana (turno matutino), tanto en Línea A como en Línea B,
  predomina una mayor proporción de resultados "Alerta" y "Fuera de Especificación (OOS)"
  que en el resto de la semana (posible efecto de arranque de equipos tras el fin de semana).
- **H3:** La Línea A y la Línea B difieren en el producto/prueba que predomina en sus
  registros (cada línea tiene un "perfil" distinto de productos).

Estas 3 hipótesis son la columna vertebral del proyecto — exactamente como en el Proyecto
Music original (actividad por ciudad/día, patrón de géneros lunes-mañana vs viernes-noche,
y preferencia de género por ciudad).


## Etapa 0 — Entorno de trabajo (Jupyter online)

Si usas **Google Colab**, **Deepnote** o **Kaggle Notebooks**, `pandas`, `numpy` y
`matplotlib` ya vienen preinstalados — no necesitas instalar nada. La celda de abajo solo
verifica versiones y, si algo faltara, lo instala al vuelo (`!pip install`).

**Cómo subir el CSV:**
- **Colab:** panel izquierdo → ícono de carpeta → "Subir" → selecciona `cpv_farma_dataset.csv`.
  Luego se leerá con `pd.read_csv("cpv_farma_dataset.csv")`.
- **Deepnote/Kaggle:** usa la opción "Upload file" / "Add data" del entorno.


In [ ]:
# Verificación de entorno (ejecútala primero)
try:
    import pandas as pd
    import numpy as np
    import matplotlib.pyplot as plt
    print("pandas:", pd.__version__)
    print("numpy:", np.__version__)
    print("Entorno listo ✅")
except ImportError as e:
    print("Falta instalar algo, ejecuta la siguiente celda:", e)


In [ ]:
# Solo si la celda anterior marcó error, descomenta y ejecuta:
# !pip install pandas numpy matplotlib


## Etapa 1 — Descripción de los datos

**Preguntas guía (respóndelas en una celda markdown después de ejecutar el código):**
1. ¿Cuántas filas y columnas tiene el dataset?
2. ¿Qué tipo de dato (`dtype`) tiene cada columna? ¿Alguno te parece incorrecto
   (por ejemplo, una fecha/hora guardada como texto)?
3. ¿Los nombres de las columnas siguen una convención consistente (snake_case,
   sin espacios ni mayúsculas)? Anota cuáles habría que corregir.
4. A simple vista en `.head()`, ¿ves algún valor que parezca ausente, duplicado
   o inconsistente (mayúsculas/minúsculas distintas para el mismo valor)?


In [ ]:
df = pd.read_csv("cpv_farma_dataset.csv")
df.head(10)


In [ ]:
df.info()


In [ ]:
df.describe(include="all")


**Tu respuesta (Etapa 1):**

_Escribe aquí tus observaciones a las 4 preguntas guía..._


## Etapa 2 — Preprocesamiento de datos

### 2.1 Nombres de columnas

**Preguntas guía:**
- ¿Qué columnas tienen espacios extra, mayúsculas o nombres poco descriptivos?
- ¿Cómo las renombrarías siguiendo snake_case (ej. `Linea_produccion` → `linea_produccion`)?


In [ ]:
print(df.columns.tolist())


In [ ]:
# Ejemplo resuelto (nivel 1): limpiar espacios y estandarizar a snake_case/minúsculas
df.columns = df.columns.str.strip().str.lower()
print(df.columns.tolist())


### 2.2 Valores ausentes

**Preguntas guía:**
- ¿Qué columnas tienen valores ausentes y cuántos?
- En un contexto de CPV real, ¿tendría sentido **eliminar** las filas con `Equipo`
  ausente, o **imputar** un valor como `"Sin registrar"`? Justifica tu decisión
  pensando en trazabilidad de datos GMP (no se "inventan" datos de equipos en la
  industria regulada; se documenta la ausencia).
- ¿Aplicarías el mismo criterio a la columna `prueba`?


In [ ]:
df.isna().sum()


> Nota de orden: primero identifica y decide qué hacer con los ausentes; los
> ejercicios de duplicados/normalización de texto (2.3) y las hipótesis (Etapa 3)
> asumen que ya ejecutaste la limpieza de nombres de columnas de 2.1.


In [ ]:
# Trata aquí los valores ausentes según tu decisión documentada arriba



### 2.3 Duplicados e inconsistencias de texto

**Preguntas guía:**
- ¿Hay filas completamente duplicadas? ¿Cuántas?
- La columna de línea de producción, ¿tiene valores como `"linea a"` y `"Linea A"`
  que en realidad son lo mismo? ¿Cómo los normalizarías?


In [ ]:
print("Duplicados exactos:", df.duplicated().sum())
print(df['linea_produccion'].unique())


In [ ]:
# Ejemplo resuelto (nivel 1): normalizar mayúsculas/minúsculas en linea_produccion
df['linea_produccion'] = df['linea_produccion'].str.strip().str.title()
print(df['linea_produccion'].unique())

# Ejercicio para ti: elimina los duplicados exactos con df.duplicated() / df.drop_duplicates()
# y confirma con df.duplicated().sum() que quedó en 0.



**Tu respuesta (Etapa 2 — resumen de decisiones tomadas):**

_Escribe aquí qué hiciste con nombres de columnas, ausentes y duplicados, y por qué..._


## Etapa 3 — Prueba de hipótesis

### H1: La actividad difiere según el día de la semana y la línea de producción

**Preguntas guía:**
- Usando `groupby` o `pivot_table`, ¿cuántos registros hay por combinación de
  `dia` y `linea_produccion`?
- ¿Qué línea tiene más volumen total de registros?
- ¿El lunes concentra más actividad que el resto de los días? ¿Es consistente en
  ambas líneas?
- ¿Cómo lo visualizarías (barras agrupadas, línea de tendencia por día)?


In [ ]:
# Tabla dinámica: conteo de registros por dia y linea de produccion
tabla_h1 = df.pivot_table(index="dia", columns="linea_produccion", values="lote_id", aggfunc="count")
tabla_h1


In [ ]:
# Visualización H1
tabla_h1.plot(kind="bar", figsize=(8,5), title="Actividad de muestreo por día y línea")
plt.ylabel("Número de registros")
plt.tight_layout()
plt.show()


**Tu conclusión sobre H1:**

_¿Se sostiene la hipótesis? ¿Qué patrón encontraste?..._


### H2: Lunes por la mañana predominan más resultados "Alerta"/"OOS"

**Preguntas guía (para la próxima sesión):**
- ¿Cómo derivarías una columna `turno` (matutino/vespertino) a partir de `hora_muestreo`?
- Comparando la proporción de cada categoría de `resultado` en "lunes-matutino" vs.
  el resto de combinaciones día-turno, ¿se confirma un pico de Alertas/OOS?
- ¿Este patrón es igual en ambas líneas o solo en una?


In [ ]:
# Pista para la próxima sesión:
# df['hora_dt'] = pd.to_datetime(df['hora_muestreo'], format='%H:%M:%S')
# df['turno'] = np.where(df['hora_dt'].dt.hour < 14, 'matutino', 'vespertino')
# pd.crosstab([df['dia'], df['turno']], df['resultado'], normalize='index')



### H3: Línea A y Línea B difieren en el producto/prueba que predomina

**Preguntas guía (para la próxima sesión):**
- ¿Cuál es la prueba (o producto, si agregas esa columna) más frecuente en cada línea?
- ¿La diferencia es marcada o marginal?
- ¿Qué implicación tendría esto para la asignación de recursos de calidad (ej. más
  HPLC en la línea que más lo usa)?


In [ ]:
# Pista para la próxima sesión:
# df.groupby('linea_produccion')['prueba'].value_counts(normalize=True)



## Conclusiones generales (plantilla)

- **Sobre los datos:** _resume la calidad de los datos y las decisiones de limpieza..._
- **Sobre H1:** _..._
- **Sobre H2:** _pendiente para próxima sesión_
- **Sobre H3:** _pendiente para próxima sesión_
- **Siguiente paso hacia ciencia de datos más avanzada:** control estadístico de
  proceso (cartas de control con `matplotlib`/`scipy`), análisis de capacidad
  (Cp/Cpk) calculado en Python, y eventualmente un modelo predictivo simple
  (ej. regresión logística) para anticipar probabilidad de OOS según
  línea/turno/día.
